# 🛰️ Notebook 1: Exploratory Data Analysis — Crop Phenology & Multi-Sensor Fusion### Project: Days to Harvest Predictor (*Seva Bharat* / Agri-Waste Valuation)This notebook explores the multi-sensor remote sensing dataset (Sentinel-2 Optical + Sentinel-1 SAR + Open-Meteo Agrometeorology) used to predict crop harvest dates.

In [ ]:
import sysimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Path# Add project root to sys.pathPROJECT_ROOT = Path("..").resolve()sys.path.insert(0, str(PROJECT_ROOT))from config.config import PROCESSED_DIR, RAW_DIRprint(f"Project Root: {PROJECT_ROOT}")print(f"Processed Dir: {PROCESSED_DIR}")

## 1. Load Training & Processed Datasets

In [ ]:
df = pd.read_csv(PROCESSED_DIR / "training_data.csv")print(f"Total Rows: {len(df):,}")print(f"Total Columns: {len(df.columns)}")print(f"Unique Farm Points: {df['point_id'].nunique()}")print(f"Years: {df['year'].unique()}")df.head(5)

## 2. Temporal Distribution & Data SplitsWe use a strictly chronological split:- **Train (2017–2023)**: 7 historical years- **Validation (2024)**: Hyperparameter tuning- **Test (2025)**: Held-out operational test set

In [ ]:
plt.figure(figsize=(8, 4), dpi=150)sns.countplot(data=df, x="year", hue="split", palette="Set2")plt.title("Sample Distribution Across Years and Data Splits", fontsize=12, fontweight="bold")plt.xlabel("Crop Season Year")plt.ylabel("Observation Count")plt.grid(True, linestyle=":", alpha=0.6)plt.show()

## 3. Remote Sensing Vegetation Indices (NDVI, NDWI, LSWI, REP, BSI)Examining the seasonal curve of vegetation and canopy moisture.

In [ ]:
plt.figure(figsize=(10, 5), dpi=150)mean_indices = df.groupby("doy")[["ndvi", "ndwi", "lswi", "bsi"]].mean().reset_index()plt.plot(mean_indices["doy"], mean_indices["ndvi"], label="NDVI (Chlorophyll/Biomass)", color="forestgreen", lw=2.2)plt.plot(mean_indices["doy"], mean_indices["ndwi"], label="NDWI (Canopy Moisture)", color="dodgerblue", lw=2.0)plt.plot(mean_indices["doy"], mean_indices["lswi"], label="LSWI (Surface Water)", color="navy", lw=1.8, linestyle="--")plt.plot(mean_indices["doy"], mean_indices["bsi"], label="BSI (Bare Soil)", color="saddlebrown", lw=1.8, linestyle=":")plt.axvline(240, color="green", linestyle=":", label="Peak Vegetative (DOY ~240)")plt.axvline(295, color="red", linestyle="--", label="Harvest Drop (DOY ~295)")plt.title("Mean Seasonal Phenology Profile across North Indian Paddy Fields", fontsize=12, fontweight="bold")plt.xlabel("Day of Year (DOY)")plt.ylabel("Index Value")plt.legend(frameon=True)plt.grid(True, linestyle=":", alpha=0.6)plt.show()

## 4. Sentinel-1 SAR Backscatter DynamicsCloud-penetrating radar shows volume scattering collapse upon harvest.

In [ ]:
plt.figure(figsize=(10, 4.5), dpi=150)mean_sar = df.groupby("doy")[["VH", "VV", "vh_vv_ratio"]].mean().reset_index()plt.plot(mean_sar["doy"], mean_sar["VH"], label="SAR VH Backscatter (dB)", color="darkorange", lw=2.2)plt.plot(mean_sar["doy"], mean_sar["VV"], label="SAR VV Backscatter (dB)", color="purple", lw=2.0)plt.title("Sentinel-1 SAR C-Band Backscatter Trajectory across Paddy Season", fontsize=12, fontweight="bold")plt.xlabel("Day of Year (DOY)")plt.ylabel("Backscatter (dB)")plt.legend(frameon=True)plt.grid(True, linestyle=":", alpha=0.6)plt.show()

## 5. Agrometeorology: GDD & VPDThermal time (Growing Degree Days) drives biological maturity.

In [ ]:
plt.figure(figsize=(10, 4.5), dpi=150)if "gdd_daily" in df.columns:    mean_wx = df.groupby("doy")[["gdd_daily", "vpd"]].mean().reset_index()    plt.plot(mean_wx["doy"], mean_wx["gdd_daily"], label="Daily GDD (Thermal Units)", color="crimson", lw=2)    plt.title("Seasonal Growing Degree Days (GDD) Profile", fontsize=12, fontweight="bold")    plt.xlabel("Day of Year (DOY)")    plt.ylabel("GDD (°C-days)")    plt.legend()    plt.grid(True, linestyle=":", alpha=0.6)    plt.show()